In [65]:
import re
import pandas as pd
import os
from pandas.core.frame import DataFrame

def read_and_parse(file_path: str, company_code: str, pattern_str: str = r'/([^/]+.(sh|cmd))') -> DataFrame:
    pattern = re.compile(pattern_str, re.I)
    data = []
    with open(file_path, 'r') as file:
        for line in file:
            if line.startswith('#'):#bypass comment
                continue
            match = pattern.search(line)
            if match:
                data.append((company_code, match.group(1), os.path.basename(file_path)))
    # final_data = set(data)
    df = pd.DataFrame(data, columns=['company','shell_name','cron_location'])

    return df

### edp.cron & edi.cron中的shells

In [66]:
from opengrok_util import codescan

df_edp_cron = pd.concat([read_and_parse('/Users/shihxuancheng/Downloads/richard-s/edp.cron.cld5', 'WHL'),
                     read_and_parse('/Users/shihxuancheng/Downloads/richard-s/edp.cron.ialcld5', 'IAL')],
                    ignore_index=True)

df_edi_cron = pd.concat([read_and_parse('/Users/shihxuancheng/Downloads/richard-s/edi.cron.cld5', 'WHL'),
                     read_and_parse('/Users/shihxuancheng/Downloads/richard-s/edi.cron.ialcld5', 'IAL')],
                    ignore_index=True)
df_ttl_cron = pd.concat([df_edp_cron, df_edi_cron], ignore_index=True)
df_ttl_cron



,company,shell_name,cron_location
0,WHL,csvsl0032.sh,edp.cron.cld5
1,WHL,csvsl0033.sh,edp.cron.cld5
2,WHL,csvsl0035.sh,edp.cron.cld5
3,WHL,csvsl0045.sh,edp.cron.cld5
4,WHL,csvsl6343.sh,edp.cron.cld5
...,...,...,...
1277,IAL,csedi2315.sh,edi.cron.ialcld5
1278,IAL,IFTMIN.sh,edi.cron.ialcld5
1279,IAL,csedi3014.sh,edi.cron.ialcld5
1280,IAL,csedimy004.sh,edi.cron.ialcld5


### VSC中內容未包含cssss0003.sh的shells

In [67]:

param_cssss0003 = [
    ('full', '-"cssss0003.sh"'),
    ('path', '(+". sh" OR +". cmd") -/Ushell')
]

records = codescan.scan(param_cssss0003, fetch_all=True)
if records:
    pattern = re.compile(r'/([^/]+.(sh|cmd))', re.I)
    data = []
    for key in records.keys():
        match = pattern.search(key)
        if match:
            system = re.search(r"^/([^/]+)/", key, re.I).group(1)
            if re.match(r'.*/ial/.*', key, re.I):
                data.append(('IAL',system, key,  match.group(1)))
            else:
                data.append(('WHL',system, key,  match.group(1)))

df_all_shells = pd.DataFrame(data, columns=['company','system', 'svn_path', 'shell_name'])
df_all_shells

Fetching: 100%|███████████████| 2571/2571 records 


,company,system,svn_path,shell_name
0,WHL,SKD,/SKD/sisapp/whl/Lshell/unused/casp2tdr_sql.cmd,casp2tdr_sql.cmd
1,IAL,ECS,/ECS/sisapp/IAL/Lshell/file2mr.cmd,file2mr.cmd
2,IAL,CMR,/CMR/sisapp/IAL/Lshell/cmr_week.sh,cmr_week.sh
3,IAL,SKD,/SKD/sisapp/IAL/Lshell/skdweb.cmd,skdweb.cmd
4,WHL,ARS,/ARS/sisapp/whl/Lshell/ars4005_1.sh,ars4005_1.sh
...,...,...,...,...
2566,WHL,BKG,/BKG/sisapp/whl/Source/cn_d2k/CAN_ED放櫃/讀碼頭edi/...,search_file.sh
2567,WHL,BKG,/BKG/sisapp/whl/Source/cn_d2k/CAN_ED放櫃/讀碼頭edi/...,getfile.sh
2568,WHL,WBA,/WBA/web/WebServerLocalGit/UATJDK1.8/CS/tpewcs...,web_shutdown_notice.sh
2569,WHL,WBA,/WBA/web/WebServerLocalGit/UATJDK1.8/CS/tpewcs...,sysDiskInfo.sh


### 存在於edp.cron & edi.cron但未包含cssss0003.sh的shells

In [69]:
import os

df_final_results = pd.merge(df_all_shells.drop_duplicates(subset=['svn_path','company','shell_name']),
                            df_ttl_cron.drop_duplicates(subset=['company','shell_name','cron_location'])
                            , on=['company','shell_name'])
df_final_results = df_final_results.loc[df_final_results['company'] == 'WHL'].sort_values(by=['company','system'])

export_file=os.path.curdir+os.sep+'scan_result.xlsx'
# df_final_results.groupby('system').size().reset_index(name='Size')

df_final_results.to_excel(export_file, sheet_name='shells', index=False)

df_final_results

,company,system,svn_path,shell_name,cron_location
174,WHL,AMSCMS,/AMSCMS/sisapp/AMS/whl/Lshell/cscms0078.sh,cscms0078.sh,edp.cron.cld5
103,WHL,BKG,/BKG/sisapp/whl/LocalObject/IALTYO5/csbkg2121.sh,csbkg2121.sh,edp.cron.cld5
121,WHL,BKG,/BKG/sisapp/whl/Lshell/IALTPE5/bkgrm.sh,bkgrm.sh,edp.cron.cld5
175,WHL,BKG,/BKG/sisapp/whl/Lshell/IAS/bkgrm.sh,bkgrm.sh,edp.cron.cld5
288,WHL,BKG,/BKG/sisapp/whl/Lshell/IALTPE5/web/csbkg2074.sh,csbkg2074.sh,edp.cron.cld5
...,...,...,...,...,...
134,WHL,WAS,/WAS/sisapp/whl/Lshell/cswas0009.sh,cswas0009.sh,edp.cron.cld5
157,WHL,WAS,/WAS/sisapp/whl/Lshell/cswas0001.sh,cswas0001.sh,edp.cron.cld5
159,WHL,WAS,/WAS/sisapp/whl/Lshell/cswas0002.sh,cswas0002.sh,edp.cron.cld5
170,WHL,WAS,/WAS/sisapp/whl/Lshell/cswas0003.sh,cswas0003.sh,edp.cron.cld5


### Test Block

In [70]:
import pandas as pd
import re
list_a = [('BKG','A','cs1111.sh'), ('BKG','B','cs1112.sh'), ('BKG','C','cs1113.sh'),('BKG','D','cs1113.sh'), ('BKG','E','cs1114.sh')]
df_a = pd.DataFrame(list_a, columns=['system','Label','shell_name'])

list_b = ['cs1112.sh', 'cs1113.sh', 'cs1114.sh', 'cs1115.sh']
df_b = pd.DataFrame(list_b, columns=['shell_name'])

df_c = pd.merge(df_a,df_b, on=['shell_name'])
df_c


,system,Label,shell_name
0,BKG,B,cs1112.sh
1,BKG,C,cs1113.sh
2,BKG,D,cs1113.sh
3,BKG,E,cs1114.sh
